# Train Models with datasets A and B and evaluate using independent dataset

In [ ]:
import pandas as pd
from DerivModel import FeedForwardNetwork, train_model
from DerivPlots import scatter_plot
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.nn.functional as F
from tqdm.notebook import tqdm, trange

## Load and Prepare the dataset A

Here dataset A will be used with the Monte Carlo samples at 100,000 in the underlying basket option model. 

In [ ]:
filename_root = 'test_basket_mt'

In [ ]:
df1 = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1.columns:
        df1[col] = (
            df1[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

In [ ]:
df1 = df1[df1['samples'] == 100000]

In [ ]:
df1 = df1.drop('samples', axis=1)

## Train model with 4 layers of width 300 for training set A

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
y = df1['option_value']

In [ ]:
X = df1.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.01) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 4096
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
epochs = 200
lr = 0.001

In [ ]:
modelA = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizer = optim.Adam(modelA.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
modelA_out = train_model(modelA, data_loader, test_data_loader, loss_fn, optimizer, epochs)

## Load and Prepare the dataset B

In [ ]:
filename_root_B = 'test_basket_mt_B'

df1B = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1B.columns:
        df1B[col] = (
            df1B[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )

df1B = df1B[df1B['samples'] == 100000]

df1B = df1B.drop('samples', axis=1)

## Train model with 4 layers of width 300 for training set B

In [ ]:
yB = df1B['option_value']

XB = df1B.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

X_trainB, X_testB, y_trainB, y_testB = train_test_split(XB, yB, test_size=0.01) 
scalerB = StandardScaler()
X_trainB = scalerB.fit_transform(X_trainB)
X_testB = scalerB.transform(X_testB)

torch_tensorB = torch.from_numpy(X_trainB).float().to(device)
y_tensorB = torch.from_numpy(y_trainB.to_numpy().copy()).float().to(device)
datasetB = TensorDataset(torch_tensorB, y_tensorB)
batch_size = 4096
data_loaderB = DataLoader(datasetB, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensorB = torch.from_numpy(X_testB).float().to(device)
test_y_tensorB = torch.from_numpy(y_testB.to_numpy().copy()).float().to(device)
test_datasetB = TensorDataset(test_torch_tensorB, test_y_tensorB)
test_data_loaderB = DataLoader(test_datasetB, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
modelB = FeedForwardNetwork(input_size=28, 
                            num_hidden_layers=4, 
                            neurons_per_layer=300).to(device)
optimizerB = optim.Adam(modelB.parameters(), lr=lr)
loss_fnB = torch.nn.MSELoss()

In [ ]:
model_out_B = train_model(modelB, data_loaderB, test_data_loaderB, loss_fnB, optimizerB, epochs)

## Load and Prepare the accuracy test dataset

In [ ]:
filename_root_acc = 'test_basket_accuracy'

df1acc = pd.concat(
    [pd.read_csv(f"{filename_root}{i}.csv") for i in range(20)],
    ignore_index=False
)

# Columns that need cleaning
cols_to_fix = ["maturity", "option_value", "error_estimate", "process_time", "samples"]

for col in cols_to_fix:
    if col in df1acc.columns:
        df1acc[col] = (
            df1acc[col]
            .astype(str)              # ensure string
            .str.strip("[]")         # remove brackets
            .astype(float)           # convert to numeric
        )


In [ ]:
df1acc = df1acc[df1acc['samples'] == 100000]

In [ ]:
df1acc

In [ ]:
df1acc = df1acc.drop('samples', axis=1)

In [ ]:
df1acc

In [ ]:
yacc = df1acc['option_value']

Xacc = df1acc.drop(['option_value', 'process_time', 'error_estimate'], axis=1)

X_acc = scaler.transform(Xacc)

test_torch_tensoracc = torch.from_numpy(X_acc).float().to(device)
test_y_tensoracc = torch.from_numpy(yacc.to_numpy().copy()).float().to(device)
test_datasetacc = TensorDataset(test_torch_tensoracc, test_y_tensoracc)
test_data_loaderacc = DataLoader(test_datasetacc, batch_size=1000, shuffle=False, drop_last=True)

In [ ]:
X_accB = scalerB.transform(Xacc)

test_torch_tensoraccB = torch.from_numpy(X_accB).float().to(device)
test_y_tensoraccB = torch.from_numpy(yacc.to_numpy().copy()).float().to(device)
test_datasetaccB = TensorDataset(test_torch_tensoraccB, test_y_tensoraccB)
test_data_loaderaccB = DataLoader(test_datasetaccB, batch_size=1000, shuffle=False, drop_last=True)

## Evaluate the models

In [ ]:
modelA.eval()
test_mse_listA = list()
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderacc:
        test_outputs = modelA(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        test_mse_listA.append(mse)

In [ ]:
test_avg_mse_A = sum(test_mse_listA) / len(test_mse_listA)
print(test_avg_mse_A)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelA_scatter')

In [ ]:
modelB.eval()
test_mse_listB = list()
with torch.no_grad():
    for batch_X, batch_y in test_data_loaderaccB:
        test_outputs = modelB(batch_X)
        mse = F.mse_loss(test_outputs.squeeze(), batch_y).item()
        test_mse_listB.append(mse)
                
test_avg_mse_B = sum(test_mse_listB) / len(test_mse_listB)
print(test_avg_mse_B)

In [ ]:
scatter_plot(test_outputs, batch_y, 'modelB_scatter')